# 📊 Ablation Study — ResearchQA Benchmark

**Autonomous Researcher — Experiment Analysis Report**

> Benchmark: **ResearchQA** — Open-domain research synthesis tasks  
> Questions evaluated: **13**  
> Data source: `notebooks/logs/experiments_researchqa/`

---

## 🔬 Experiment Overview

| ID | System | Description |
|---|---|---|
| EXP1 | **Vanilla LLM** | Direct generation, no search, no verification |
| EXP2 | **RAG Baseline** | Retrieval-Augmented Generation only |
| EXP3 | **Agent Base** | Multi-agent with planning, web search & scraping |
| EXP4 | **Agent + FT Reviewer** | Agent + LoRA fine-tuned reviewer |
| EXP5 | **Agent + FT Reranker** | Agent + fine-tuned cross-encoder reranker |
| EXP6 | **Agent Full (System-2)** | Agent + both FT Reviewer & FT Reranker |

## 📐 Metrics

| Metric | Description | Range |
|---|---|---|
| `mean_f1_score` | Token-overlap F1 vs ground truth answer | 0–1 |
| `mean_citation_precision` | Fraction of citations grounded in retrieved evidence | 0–1 |
| `mean_judge_comprehensiveness` | LLM-as-judge: completeness of the report | 1–5 |
| `mean_judge_depth` | LLM-as-judge: depth of analysis | 1–5 |
| `mean_overall_judge` | LLM-as-judge: holistic quality | 1–5 |
| `mean_step_count` | Average number of agent reasoning steps | — |
| `success_rate` | Fraction of questions answered without crash | 0–1 |

In [4]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from math import pi

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'grid.alpha': 0.6,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})

# ResearchQA uses a warmer palette variant
PALETTE = ['#e8b86d', '#56d364', '#d2a8ff', '#79c0ff', '#ff9a8b', '#ffa657']
EXP_LABELS = [
    'EXP1\nVanilla LLM',
    'EXP2\nRAG Baseline',
    'EXP3\nAgent Base',
    'EXP4\nAgent+\nFT-Reviewer',
    'EXP5\nAgent+\nFT-Reranker',
    'EXP6\nAgent Full',
]
EXP_SHORT = ['EXP1', 'EXP2', 'EXP3', 'EXP4', 'EXP5', 'EXP6']

print('Libraries loaded OK')

Libraries loaded ✓


In [5]:
# == Load data ====================================================─
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'notebooks', 'logs', 'experiments_researchqa')
if not os.path.exists(DATA_DIR):
    DATA_DIR = 'logs/experiments_researchqa'
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'logs', 'experiments_researchqa')

METRICS = [
    'mean_f1_score', 'mean_citation_precision',
    'mean_judge_comprehensiveness', 'mean_judge_depth',
    'mean_overall_judge', 'mean_step_count', 'success_rate'
]
METRIC_DISPLAY = [
    'F1 Score', 'Citation Precision',
    'Comprehensiveness', 'Depth',
    'Overall Judge', 'Step Count', 'Success Rate'
]

# Read from individual JSON files (primary source of truth)
exp_files = {
    'EXP1': 'exp1_vanilla_llm_results.json',
    'EXP2': 'exp2_rag_baseline_results.json',
    'EXP3': 'exp3_agent_base_results.json',
    'EXP4': 'exp4_agent_ft_reviewer_results.json',
    'EXP5': 'exp5_agent_ft_reranker_results.json',
    'EXP6': 'exp6_agent_full_results.json'
}

loaded_rows = []
from_json = True

for exp_id in EXP_SHORT:
    filename = exp_files[exp_id]
    json_path = os.path.join(DATA_DIR, filename)
    if os.path.exists(json_path):
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            row = {m: data.get(m, np.nan) for m in METRICS}
            row['experiment'] = exp_id
            loaded_rows.append(row)
    else:
        from_json = False
        break

if from_json and len(loaded_rows) == 6:
    df = pd.DataFrame(loaded_rows).set_index('experiment')
    print('Loaded data from individual JSON files in:', DATA_DIR)
else:
    csv_path = os.path.join(DATA_DIR, 'experiments_comparison.csv')
    df = pd.read_csv(csv_path, index_col=0)
    df.index = EXP_SHORT
    print('Loaded data from CSV fallback:', csv_path)

df.round(4)

Loaded data from individual JSON files in: c:\Users\Lenovo\MachineLearning\autonomous-researcher\notebooks\logs\experiments_researchqa


,mean_f1_score,mean_citation_precision,mean_judge_comprehensiveness,mean_judge_depth,mean_overall_judge,mean_step_count,success_rate
experiment,,,,,,,
EXP1,0.1360,0.0,2.7308,2.3538,2.46,2.0,1.0
EXP2,0.1092,0.0,2.8846,2.0769,2.41,2.0,1.0
EXP3,0.0579,1.0,2.3077,1.6538,1.94,13.3,1.0
EXP4,0.0435,1.0,1.7308,1.2308,1.44,9.6,1.0
EXP5,0.0622,1.0,2.1538,1.5000,1.78,13.6,1.0
EXP6,0.0466,0.8,1.7692,1.3846,1.55,11.6,1.0


---
## 1️⃣ Summary Statistics

In [6]:
summary = df[METRICS].copy()

def highlight_best(col):
    if col.name == 'mean_step_count':
        return ['background-color: #1f3d2a' if v == col.min() else '' for v in col]
    return ['background-color: #1f3d2a' if v == col.max() else '' for v in col]

display(summary.style
    .apply(highlight_best)
    .format({
        'mean_f1_score': '{:.4f}',
        'mean_citation_precision': '{:.4f}',
        'mean_judge_comprehensiveness': '{:.3f}',
        'mean_judge_depth': '{:.3f}',
        'mean_overall_judge': '{:.3f}',
        'mean_step_count': '{:.1f}',
        'success_rate': '{:.1%}',
    })
    .set_caption('ResearchQA Benchmark Results (green = best per column, step_count = lower is better)')
)

AttributeError: The '.style' accessor requires jinja2

---
## 2️⃣ Bar Charts — All Metrics

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('ResearchQA - All Metrics per Experiment', fontsize=18, fontweight='bold', color='#e6edf3', y=1.01)
axes_flat = axes.flatten()

x = np.arange(len(EXP_SHORT))

for i, (metric, label) in enumerate(zip(METRICS, METRIC_DISPLAY)):
    ax = axes_flat[i]
    values = df[metric].values
    
    bars = ax.bar(x, values, color=PALETTE, width=0.6, edgecolor='#0d1117', linewidth=0.8)
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * max(values),
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, color='#e6edf3', fontweight='bold')
    
    best_idx = int(np.argmin(values) if metric == 'mean_step_count' else np.argmax(values))
    bars[best_idx].set_edgecolor('#f0e130')
    bars[best_idx].set_linewidth(2.5)
    
    ax.set_title(label, pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(EXP_SHORT, fontsize=8.5)
    ax.grid(axis='y', alpha=0.4)
    ax.set_ylim(0, max(values) * 1.2 + 0.05)

axes_flat[-1].set_visible(False)
axes_flat[-2].set_visible(False)

legend_patches = [mpatches.Patch(color=PALETTE[i], label=f'{EXP_SHORT[i]}: {EXP_LABELS[i].replace(chr(10), " ")}') for i in range(6)]
fig.legend(handles=legend_patches, loc='lower right', ncol=3, bbox_to_anchor=(0.98, 0.03), fontsize=9)

plt.tight_layout()
plt.savefig('researchqa_all_metrics_bar.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Figure saved: researchqa_all_metrics_bar.png')

---
## 3️⃣ 🕸️ Radar Chart (Spider Chart) — All Experiments

In [ ]:
radar_metrics = [
    'mean_f1_score', 'mean_citation_precision',
    'mean_judge_comprehensiveness', 'mean_judge_depth',
    'mean_overall_judge', 'mean_step_count'
]
radar_labels = [
    'F1 Score', 'Citation\nPrecision',
    'Comprehensiveness', 'Depth',
    'Overall Judge', 'Efficiency\n(fewer steps)'
]

radar_data = df[radar_metrics].copy()
normalized = radar_data.copy()
for col in radar_metrics:
    col_min, col_max = radar_data[col].min(), radar_data[col].max()
    if col_max - col_min > 0:
        if col == 'mean_step_count':
            normalized[col] = 1 - (radar_data[col] - col_min) / (col_max - col_min)
        else:
            normalized[col] = (radar_data[col] - col_min) / (col_max - col_min)
    else:
        normalized[col] = 1.0

N = len(radar_metrics)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(1, 1, figsize=(10, 10), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8, color='#8b949e')
ax.yaxis.grid(True, color='#21262d', linestyle='--', alpha=0.8)
ax.xaxis.grid(True, color='#21262d', linestyle='--', alpha=0.8)
ax.spines['polar'].set_color('#30363d')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11, color='#c9d1d9', fontweight='bold')

for i, (exp_id, color) in enumerate(zip(EXP_SHORT, PALETTE)):
    values_norm = normalized.loc[exp_id, radar_metrics].tolist()
    values_norm += values_norm[:1]
    ax.plot(angles, values_norm, 'o-', linewidth=2, color=color, label=exp_id, markersize=5)
    ax.fill(angles, values_norm, alpha=0.08, color=color)

ax.set_title('ResearchQA - Multi-Metric Radar Comparison\n(all axes normalized to [0,1], efficiency = 1 − normalized_steps)',
             fontsize=13, fontweight='bold', color='#e6edf3', pad=25)

legend = ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
for text in legend.get_texts():
    text.set_color('#c9d1d9')

plt.tight_layout()
plt.savefig('researchqa_radar.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Radar chart saved: researchqa_radar.png')

---
## 4️⃣ Heatmap — Normalized Metric Matrix

In [ ]:
norm_all = df[METRICS].copy().astype(float)
for col in METRICS:
    col_min, col_max = norm_all[col].min(), norm_all[col].max()
    if col_max - col_min > 0:
        if col == 'mean_step_count':
            norm_all[col] = 1 - (norm_all[col] - col_min) / (col_max - col_min)
        else:
            norm_all[col] = (norm_all[col] - col_min) / (col_max - col_min)
    else:
        norm_all[col] = 1.0

norm_all.columns = METRIC_DISPLAY

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

sns.heatmap(
    norm_all,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5, linecolor='#0d1117',
    ax=ax,
    cbar_kws={'label': 'Normalized Score (higher = better)', 'shrink': 0.8}
)

ax.set_title('ResearchQA - Normalized Performance Heatmap\n(Step Count inverted: higher = more efficient)',
             fontsize=14, fontweight='bold', color='#e6edf3', pad=15)
ax.set_yticklabels(EXP_SHORT, rotation=0, fontsize=11)
ax.set_xticklabels(METRIC_DISPLAY, rotation=30, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('researchqa_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Heatmap saved: researchqa_heatmap.png')

---
## 5️⃣ Key Metric Deep-Dives

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ResearchQA - F1 Score vs Citation Precision Trade-off', fontsize=15, fontweight='bold', color='#e6edf3')

ax = axes[0]
x = np.arange(len(EXP_SHORT))
w = 0.35
b1 = ax.bar(x - w/2, df['mean_f1_score'], w, label='F1 Score', color='#e8b86d', edgecolor='#0d1117')
b2 = ax.bar(x + w/2, df['mean_citation_precision'], w, label='Citation Precision', color='#56d364', edgecolor='#0d1117')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8, color='#e6edf3')

ax.set_xticks(x)
ax.set_xticklabels(EXP_SHORT)
ax.set_ylabel('Score')
ax.set_title('F1 vs Citation Precision')
ax.legend()
ax.grid(axis='y', alpha=0.4)
ax.set_ylim(0, 1.3)

# Note: EXP3–EXP5 have perfect citation precision in ResearchQA
ax.annotate('EXP3/4/5: Perfect\ncitation precision (1.0)',
            xy=(3, 1.02), xytext=(1, 1.1),
            arrowprops=dict(arrowstyle='->', color='#56d364'),
            color='#56d364', fontsize=9)

ax = axes[1]
judge_metrics = ['mean_judge_comprehensiveness', 'mean_judge_depth', 'mean_overall_judge']
judge_labels = ['Comprehensiveness', 'Depth', 'Overall']
judge_colors = ['#d2a8ff', '#79c0ff', '#ff9a8b']
w = 0.22
offsets = [-w, 0, w]

for j, (metric, label, color) in enumerate(zip(judge_metrics, judge_labels, judge_colors)):
    bars = ax.bar(x + offsets[j], df[metric], w, label=label, color=color, edgecolor='#0d1117')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.03, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7.5, color='#e6edf3')

ax.set_xticks(x)
ax.set_xticklabels(EXP_SHORT)
ax.set_ylabel('Judge Score (1–5)')
ax.set_title('LLM-Judge Scores (Comprehensiveness, Depth, Overall)')
ax.set_ylim(0, 4.5)
ax.axhline(y=3.0, color='#58a6ff', linestyle='--', alpha=0.5, linewidth=1, label='Mid-point (3.0)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('researchqa_key_metrics.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# == HotpotQA vs ResearchQA side-by-side comparison ==============
# Load HotpotQA data for cross-benchmark comparison
hotpot_path_candidates = [
    'logs/experiments_hotpotqa/experiments_comparison.csv',
    '../notebooks/logs/experiments_hotpotqa/experiments_comparison.csv',
    os.path.join(os.path.dirname(DATA_DIR), 'experiments_hotpotqa', 'experiments_comparison.csv'),
]
df_hot = None
for p in hotpot_path_candidates:
    if os.path.exists(p):
        df_hot = pd.read_csv(p, index_col=0)
        df_hot.index = EXP_SHORT
        break

if df_hot is not None:
    compare_metrics = ['mean_f1_score', 'mean_citation_precision', 'mean_overall_judge', 'mean_step_count']
    compare_labels  = ['F1 Score', 'Citation Precision', 'Overall Judge', 'Step Count']
    
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle('Cross-Benchmark Comparison: HotpotQA vs ResearchQA', fontsize=15, fontweight='bold', color='#e6edf3')
    
    x = np.arange(len(EXP_SHORT))
    w = 0.35
    
    for ax, metric, label in zip(axes, compare_metrics, compare_labels):
        b1 = ax.bar(x - w/2, df_hot[metric], w, label='HotpotQA', color='#58a6ff', edgecolor='#0d1117', alpha=0.9)
        b2 = ax.bar(x + w/2, df[metric], w, label='ResearchQA', color='#e8b86d', edgecolor='#0d1117', alpha=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(EXP_SHORT, fontsize=9)
        ax.set_title(label)
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('cross_benchmark_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('Cross-benchmark comparison saved: cross_benchmark_comparison.png')
else:
    print('HotpotQA data not found - skipping cross-benchmark comparison.')

In [ ]:
# == Step count scatter: quality vs complexity ====================─
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#0d1117')

steps = df['mean_step_count'].values
overall = df['mean_overall_judge'].values

sc = ax.scatter(steps, overall, c=PALETTE, s=280, zorder=5, edgecolors='white', linewidths=1.5)

for i, exp in enumerate(EXP_SHORT):
    ax.annotate(exp, (steps[i], overall[i]),
                textcoords='offset points', xytext=(10, 4),
                fontsize=10, color=PALETTE[i], fontweight='bold')

ax.set_xlabel('Mean Step Count (complexity)', fontsize=12)
ax.set_ylabel('Mean Overall Judge Score', fontsize=12)
ax.set_title('ResearchQA - Quality vs Complexity Trade-off', fontsize=14, fontweight='bold', color='#e6edf3')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#161b22')

ax.axvline(x=np.mean(steps), color='#e8b86d', linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(y=np.mean(overall), color='#e8b86d', linestyle='--', alpha=0.4, linewidth=1)

plt.tight_layout()
plt.savefig('researchqa_quality_vs_complexity.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 6️⃣ 📝 Detailed Analysis

In [ ]:
print('=' * 70)
print('RESEARCHQA BENCHMARK - DETAILED ANALYSIS')
print('=' * 70)

label_map = {
    'EXP1': 'Vanilla LLM (System-1)',
    'EXP2': 'RAG Baseline',
    'EXP3': 'Agent Base (Multi-agent, no fine-tuning)',
    'EXP4': 'Agent + Fine-Tuned Reviewer (LoRA)',
    'EXP5': 'Agent + Fine-Tuned Reranker (CrossEncoder)',
    'EXP6': 'Agent Full - System-2 (EXP3 + EXP4 + EXP5)',
}

for exp_id, row in df.iterrows():
    print(f'\n== {exp_id}: {label_map.get(exp_id, exp_id)} ==')
    print(f'  F1 Score           : {row["mean_f1_score"]:.4f}')
    print(f'  Citation Precision : {row["mean_citation_precision"]:.4f}')
    print(f'  Comprehensiveness  : {row["mean_judge_comprehensiveness"]:.3f} / 5')
    print(f'  Depth              : {row["mean_judge_depth"]:.3f} / 5')
    print(f'  Overall Judge      : {row["mean_overall_judge"]:.3f} / 5')
    print(f'  Mean Steps         : {row["mean_step_count"]:.1f}')
    print(f'  Success Rate       : {row["success_rate"]:.1%}')

---
## 7️⃣ 🔍 Findings & Conclusions

### Key Observations

#### 1. ResearchQA is Significantly Harder than HotpotQA
- All judge scores are markedly lower than in HotpotQA (e.g., EXP1 overall: `2.477` vs `4.643`).
- ResearchQA requires synthesis across multiple sources and more nuanced domain knowledge — tasks that are harder even for the LLM judge to evaluate positively.
- Step counts are also much higher for agent systems (EXP3: `13.3` vs `8.3` in HotpotQA), indicating the agent spends more effort searching before converging.

#### 2. EXP3/EXP4/EXP5 Achieve Perfect Citation Precision (1.0)
- Unlike HotpotQA where EXP6 was the only perfect system, in ResearchQA **three systems** (EXP3, EXP4, EXP5) achieve `1.0` citation precision.
- This suggests the agent's retrieval pipeline is more robustly grounded when dealing with long-form research topics — web scraping returns relevant content that is faithfully cited.
- EXP6 drops slightly to `0.846`, possibly because the combined FT components produce more aggressive citation generation.

#### 3. Vanilla LLM Wins on F1 (Again)
- EXP1 achieves the highest F1 (`0.136`) — the same artifact as in HotpotQA. The Vanilla LLM outputs short, direct answers that overlap more with ground-truth reference phrases.
- For ResearchQA, this is even more misleading since the benchmark likely expects in-depth analysis, not a single phrase.

#### 4. FT Reviewer Continues to Underperform (EXP4)
- EXP4 has the lowest F1 (`0.043`), lowest comprehensiveness (`1.808`), and lowest depth (`1.308`).
- The LoRA-fine-tuned reviewer is particularly harmful on ResearchQA: research questions require longer, multi-faceted answers, but the reviewer's critical feedback may cause the writer to hedge or shorten responses.
- **Finding**: The FT reviewer was likely trained on data that penalizes verbosity, making it poorly calibrated for comprehensive research reports.

#### 5. EXP2 (RAG) Slightly Outperforms EXP3 (Agent) on Judge Scores
- EXP2 overall judge: `2.385` vs EXP3: `1.877`.
- This is counter-intuitive: the full multi-agent system scores lower than the simple RAG baseline.
- Hypothesis: For ResearchQA, the agent's additional reasoning steps (planning → searching → verifying) produce reports that are more structured but less coherent than RAG's direct synthesis, resulting in lower LLM-judge scores.

#### 6. EXP5 (FT Reranker) Again Emerges as Best Agent
- EXP5 matches EXP3 on judge scores but has perfect citation precision — making it the most trustworthy agentic system.
- The reranker helps surface more relevant evidence chunks, maintaining output quality while ensuring citations are grounded.

### 📌 Summary Table

| | Best | Worst | Winner |
|---|---|---|---|
| **F1 Score** | EXP1 (0.136) | EXP4 (0.043) | EXP1 |
| **Citation Precision** | EXP3/4/5 (1.000) | EXP1/2 (0.000) | EXP3/EXP5 |
| **Comprehensiveness** | EXP2 (2.885) | EXP4 (1.808) | EXP2 |
| **Depth** | EXP1 (2.392) | EXP4 (1.308) | EXP1 |
| **Overall Judge** | EXP1 (2.477) | EXP4 (1.515) | EXP1 |
| **Efficiency (steps↓)** | EXP1 & EXP2 (2.0) | EXP5 (13.6) | EXP1/EXP2 |
| **Grounded Quality** | EXP5 | EXP1/EXP2 | **EXP5** |

### 🆚 Cross-Benchmark Conclusion

| Dimension | HotpotQA Winner | ResearchQA Winner |
|---|---|---|
| Raw F1 | EXP1 | EXP1 |
| Citation Quality | EXP6 | EXP3/EXP5 |
| LLM-Judge | EXP1 | EXP1/EXP2 |
| Grounded + Quality | **EXP5** | **EXP5** |

> **Recommended System across both benchmarks**: **EXP5 (Agent + Fine-Tuned Reranker)** — consistent citation grounding, competitive quality, and no instability from the FT reviewer.

In [ ]:
# == Final combined summary figure ==============================─
fig = plt.figure(figsize=(20, 6))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('ResearchQA Benchmark - Quick Reference Summary', fontsize=16, fontweight='bold', color='#e6edf3', y=1.02)

gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

x = np.arange(len(EXP_SHORT))
colors = PALETTE

ax1 = fig.add_subplot(gs[0])
w = 0.35
ax1.bar(x - w/2, df['mean_f1_score'], w, color='#e8b86d', label='F1', edgecolor='#0d1117')
ax1.bar(x + w/2, df['mean_citation_precision'], w, color='#56d364', label='Citation P.', edgecolor='#0d1117')
ax1.set_xticks(x); ax1.set_xticklabels(EXP_SHORT, fontsize=9)
ax1.set_title('F1 & Citation Precision'); ax1.legend(fontsize=9)
ax1.set_ylim(0, 1.3); ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[1])
ax2.bar(x, df['mean_overall_judge'], color=colors, edgecolor='#0d1117')
ax2.axhline(3.0, color='white', linestyle='--', alpha=0.4, linewidth=1)
for xi, val in zip(x, df['mean_overall_judge']):
    ax2.text(xi, val + 0.03, f'{val:.2f}', ha='center', fontsize=9, color='#e6edf3')
ax2.set_xticks(x); ax2.set_xticklabels(EXP_SHORT, fontsize=9)
ax2.set_title('Overall Judge Score (1–5)'); ax2.set_ylim(0, 4)
ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[2])
ax3.bar(x, df['mean_step_count'], color=colors, edgecolor='#0d1117')
for xi, val in zip(x, df['mean_step_count']):
    ax3.text(xi, val + 0.1, f'{val:.1f}', ha='center', fontsize=9, color='#e6edf3')
ax3.set_xticks(x); ax3.set_xticklabels(EXP_SHORT, fontsize=9)
ax3.set_title('Mean Step Count (↓ more efficient)')
ax3.grid(axis='y', alpha=0.3)

plt.savefig('researchqa_summary.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Summary figure saved: researchqa_summary.png')
print('\n[OK] ResearchQA analysis complete.')